# Scenario Dataset Builder from Toolkit Excel
## Transport Decarbonization Pathways transferable input version

This notebook generates the same master scenario dataset as `Scenarios_Definition_Levers_ASI.ipynb`, but reads the lab inputs from `Toolkit_Mobility_Pathways_Inputs.xlsx`.

For the current Gipuzkoa workbook, `APPLY_GIP_COMPATIBILITY_PATCH=True` documents and applies the small differences needed to reproduce the historical output: taxi rows are excluded because the workbook flags them as inconsistent, `eBike` fleet is set to the value used by `Fleet.csv`, and the missing walking pkm baseline is restored from the original notebook.


In [35]:
# ===========================================================================
#  IMPORTS AND USER PARAMETERS
# ===========================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import pandas as pd

from vehicle_emissions_package import build_df_merged_final
from vehicle_emissions_package.model import DEFAULT_VEHICLE_PARAMS


In [36]:
#THIS IS WHERE YOU CHANGE THE LAB!
LAB_ID = "GIP"
BASE_YEAR = 2025
OUTPUT_STEM = "scenarios_master_toolkit"
OUT_DIR = Path(".")
PARQUET_PATH = OUT_DIR / f"{OUTPUT_STEM}.parquet"
CSV_PREVIEW  = OUT_DIR / f"{OUTPUT_STEM}_preview.csv"


#After running all the cells in the notebook, it will generate the following files in the same folder:

#  scenarios_master_boston.parquet 

#  scenarios_master_boston_preview.csv 



In [ ]:

INPUT_XLSX = Path("Toolkit_Mobility_Pathways_Inputs.xlsx")
YEARS = np.arange(BASE_YEAR, 2051)
N_YEARS = len(YEARS)


print("Environment ready.")
print("Input:", INPUT_XLSX.resolve())
print("Lab:", LAB_ID, "Base year:", BASE_YEAR)
print("Output paths:")
print("  ", PARQUET_PATH.resolve())
print("  ", CSV_PREVIEW.resolve())









# Keep True to match the current Gipuzkoa result from the original notebook.
# Set False when using a cleaned workbook for another lab.
APPLY_GIP_COMPATIBILITY_PATCH = True


Environment ready.
Input: /Users/joanpau/Desktop/Toolkit/TEST RESULTADOS /Toolkit_Mobility_Pathways_Inputs.xlsx
Lab: GIP Base year: 2025
Output paths:
   /Users/joanpau/Desktop/Toolkit/TEST RESULTADOS /scenarios_master_toolkit.parquet
   /Users/joanpau/Desktop/Toolkit/TEST RESULTADOS /scenarios_master_toolkit_preview.csv


## 1 · Methodology — the Kaya identity for passenger mobility

We compute **annual passenger-transport GHG emissions** in Gipuzkoa using a
mode-resolved Kaya decomposition:

$$
E_y \;=\; \text{Pop}_y \;\cdot\; \text{pkm}_y/\text{cap} \;\cdot\; \sum_{m \in M} s_{m,y} \cdot \big(\text{EF}^{\text{op}}_{m,y} + \text{EF}^{\text{emb}}_{m,y}\big)
$$

where

- **$\text{Pop}_y$** — population (constant 735 332 hab in this study).
- **$\text{pkm}_y/\text{cap}$** — per-capita passenger-kilometres, modulated by L₃ (demand).
- **$s_{m,y}$** — modal share of mode $m$ at year $y$, modulated by L₂ (modal shift).
- **$\text{EF}^{\text{op}}_{m,y}$** — operational emission factor (TTW + WTT), in gCO₂eq/pkm. It is a blend of fossil and electric technologies whose mix is ramped by L₁ (EV share) and whose electricity footprint is set by the grid scenario.
- **$\text{EF}^{\text{emb}}_{m,y}$** — embodied emission factor (B1 manufacturing + B2 infrastructure + B3 disposal), blended the same way, and additionally scaled by L_Embodied (embodied-LCA cut by 2050).

The two EF components are treated separately because they respond to
**different** levers: the operational part reacts to L₁ (vehicle mix) and the
grid; the embodied part reacts to L₁ and L_Embodied.

**Cumulative per-capita emissions** are simply
$\sum_{y=2025}^{2050} E_y / \text{Pop}_y$.
**Absolute cumulative emissions** are $\sum_{y=2025}^{2050} E_y$.



## 2 · Baseline 2025 — population, pkm, fleet

We keep three tables fixed at 2025 values and let the levers *evolve* them
year by year.

In [38]:
# ---------------------------------------------------------------------------
# Read baseline population, pkm by mode, and fleet from Toolkit Excel
# ---------------------------------------------------------------------------
def _find_col(columns, *needles):
    for col in columns:
        norm = str(col).lower().replace("\n", " ")
        if all(n.lower() in norm for n in needles):
            return col
    raise KeyError(f"Could not find column containing: {needles}")


def _sheet(name):
    return pd.read_excel(INPUT_XLSX, sheet_name=name, header=1)


def _parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        return float(value.replace(",", ""))
    return float(value)


def _parse_occupancy(value):
    # Excel converted some decimal occupancies to dates in the current workbook.
    # Example: intended 1.5 -> 2026-05-01; intended 15.3 -> 2026-03-15.
    if pd.isna(value):
        return np.nan
    if isinstance(value, pd.Timestamp):
        value = value.to_pydatetime()
    if isinstance(value, datetime):
        return float(f"{value.day}.{value.month}")
    if isinstance(value, str):
        return float(value.replace(",", "."))
    return float(value)


macro_raw = _sheet("macro")
fleet_raw = _sheet("fleet")
vkt_raw = _sheet("VKT and occupancy (if found)")
modal_raw = _sheet("modal_split")

macro_lab = _find_col(macro_raw.columns, "lab id")
macro_year = _find_col(macro_raw.columns, "year")
macro_pop = _find_col(macro_raw.columns, "population")

fleet_lab = _find_col(fleet_raw.columns, "lab id")
fleet_year = _find_col(fleet_raw.columns, "year")
fleet_mode = _find_col(fleet_raw.columns, "transport mode")
fleet_tech = _find_col(fleet_raw.columns, "technology")
fleet_count = _find_col(fleet_raw.columns, "fleet count")

vkt_lab = _find_col(vkt_raw.columns, "lab id")
vkt_year = _find_col(vkt_raw.columns, "year")
vkt_mode = _find_col(vkt_raw.columns, "transport mode")
vkt_vkt = _find_col(vkt_raw.columns, "vkt")
vkt_occ = _find_col(vkt_raw.columns, "occupancy")

modal_lab = _find_col(modal_raw.columns, "lab id")
modal_year = _find_col(modal_raw.columns, "year")
modal_mode = _find_col(modal_raw.columns, "transport mode")
modal_share = _find_col(modal_raw.columns, "modal share")

macro_lab_df = macro_raw[(macro_raw[macro_lab] == LAB_ID) & (macro_raw[macro_year] == BASE_YEAR)].copy()
if macro_lab_df.empty:
    raise ValueError(f"No macro row found for LAB_ID={LAB_ID!r}, BASE_YEAR={BASE_YEAR}")
POP = int(round(float(macro_lab_df.iloc[0][macro_pop])))

mode_to_modal = {"Walk": "Walking", "Walking": "Walking",
                 "Bike": "Bike", "BikeSharing": "Bike",
                 "Moped": "Moto", "Moto": "Moto",
                 "Car": "Auto", "Auto": "Auto", "Taxi": "Auto",
                 "Bus": "Bus", "Train": "Train"}
MODES = ["Walking", "Auto", "Bus", "Moto", "Bike", "Train"]
N_MODES = len(MODES)

# Baseline annual pkm by modal mode from VKT * occupancy.
vkt = vkt_raw[(vkt_raw[vkt_lab] == LAB_ID) & (vkt_raw[vkt_year] == BASE_YEAR)].copy()
vkt = vkt[[vkt_mode, vkt_vkt, vkt_occ]].dropna(subset=[vkt_mode])
vkt.columns = ["mode", "vkt", "occupancy"]
vkt["modal_mode"] = vkt["mode"].map(mode_to_modal)
vkt["vkt"] = vkt["vkt"].map(_parse_number)
vkt["occupancy"] = vkt["occupancy"].map(_parse_occupancy)
vkt["pkm"] = vkt["vkt"] * vkt["occupancy"]
pkm_by_mode_2025 = vkt.dropna(subset=["modal_mode", "pkm"]).groupby("modal_mode")["pkm"].sum().to_dict()

if APPLY_GIP_COMPATIBILITY_PATCH and LAB_ID == "GIP":
    # Exact baseline used by Scenarios_Definition_Levers_ASI.ipynb.
    # This removes tiny VKT*occupancy rounding drift and fills missing walking pkm.
    pkm_by_mode_2025 = {
        "Walking": 438_948_130,
        "Auto": 4_166_221_037,
        "Bus": 516_517_981,
        "Moto": 189_199_982,
        "Bike": 94_599_991,
        "Train": 496_677_196,
    }

fallback_notes = []
missing_modes = [m for m in MODES if m not in pkm_by_mode_2025 or not np.isfinite(pkm_by_mode_2025[m])]

# Generic transferability fallback: if VKT/occupancy is incomplete, use modal_split
# to impute missing pkm from the known pkm modes. Modal split is trip-based, so this
# is less precise than VKT*occupancy; the notebook prints the fallback explicitly.
modal = modal_raw[(modal_raw[modal_lab] == LAB_ID) & (modal_raw[modal_year] == BASE_YEAR)].copy()
modal = modal[[modal_mode, modal_share]].dropna(subset=[modal_mode])
modal.columns = ["mode", "share"]
modal["modal_mode"] = modal["mode"].map(mode_to_modal)
modal["share"] = pd.to_numeric(modal["share"], errors="coerce")
modal_share_by_mode = modal.dropna(subset=["modal_mode"]).groupby("modal_mode")["share"].sum().to_dict()

known_modes = [m for m in MODES if m in pkm_by_mode_2025 and np.isfinite(pkm_by_mode_2025[m])]
known_total_pkm = sum(float(pkm_by_mode_2025[m]) for m in known_modes)
known_total_share = sum(float(modal_share_by_mode.get(m, 0.0) or 0.0) for m in known_modes)

if missing_modes and known_total_pkm > 0 and known_total_share > 0:
    for m in missing_modes:
        share = float(modal_share_by_mode.get(m, 0.0) or 0.0)
        if share > 0:
            pkm_by_mode_2025[m] = known_total_pkm * share / known_total_share
            fallback_notes.append(
                f"{m} pkm was missing in VKT/occupancy; estimated from modal_split "
                "using the average pkm per modal-share point of known modes."
            )
        else:
            pkm_by_mode_2025[m] = 0.0
            fallback_notes.append(
                f"{m} pkm was missing and modal_split share was empty/zero; set to 0."
            )

missing_modes = [m for m in MODES if m not in pkm_by_mode_2025 or not np.isfinite(pkm_by_mode_2025[m])]
if missing_modes:
    raise ValueError(
        "Missing baseline pkm for modes: " + ", ".join(missing_modes) +
        ". Add VKT and occupancy rows or modal_split shares for these modes."
    )
pkm_by_mode_2025 = {m: float(pkm_by_mode_2025[m]) for m in MODES}
total_pkm_2025 = sum(pkm_by_mode_2025.values())

# Fleet composition -> tech-share within each mode.
fleet = fleet_raw[(fleet_raw[fleet_lab] == LAB_ID) & (fleet_raw[fleet_year] == BASE_YEAR)].copy()
fleet = fleet[[fleet_mode, fleet_tech, fleet_count]].dropna(subset=[fleet_mode, fleet_tech, fleet_count])
fleet.columns = ["mode", "technology", "fleet"]
fleet["fleet"] = pd.to_numeric(fleet["fleet"], errors="coerce")
fleet = fleet.dropna(subset=["fleet"])

if APPLY_GIP_COMPATIBILITY_PATCH and LAB_ID == "GIP":
    # Matches the original Fleet.csv used by Scenarios_Definition_Levers_ASI.ipynb.
    fleet = fleet[fleet["mode"] != "Taxi"].copy()
    fleet.loc[fleet["technology"] == "eBike", "fleet"] = 79_100

missing_tech = sorted(set(fleet["technology"]) - set(DEFAULT_VEHICLE_PARAMS))
if missing_tech:
    raise ValueError(f"Technologies not found in DEFAULT_VEHICLE_PARAMS: {missing_tech}")

fleet["annual_mileage"]  = fleet["technology"].map(lambda t: DEFAULT_VEHICLE_PARAMS[t]["annual_mileage"])
fleet["passenger_load"]  = fleet["technology"].map(lambda t: DEFAULT_VEHICLE_PARAMS[t]["passenger_load"])
fleet["vkt_fleet"]       = fleet["fleet"] * fleet["annual_mileage"]
fleet["pkm_fleet"]       = fleet["vkt_fleet"] * fleet["passenger_load"]
fleet["modal_mode"]      = fleet["mode"].map(mode_to_modal)

if fleet["modal_mode"].isna().any():
    bad_modes = sorted(fleet.loc[fleet["modal_mode"].isna(), "mode"].unique())
    raise ValueError(f"Fleet modes are not mapped to modal modes: {bad_modes}")

pkm_by_modal            = fleet.groupby("modal_mode")["pkm_fleet"].transform("sum")
fleet["tech_share_pkm"] = fleet["pkm_fleet"] / pkm_by_modal

ELECTRIC_TECHS = {"Walk", "Bike_Standalone", "eBike", "BikeSharing_Standalone",
                  "eBikeSharing", "Moped_BEV", "Car_BEV", "Taxi_BEV",
                  "Bus_BEV", "Rail_EV"}
MOTORIZED_MODES = ["Auto", "Moto", "Bus"]
def _current_ev_share():
    e, t = 0.0, 0.0
    for m in MOTORIZED_MODES:
        sub = fleet[fleet["modal_mode"] == m]
        t += sub["pkm_fleet"].sum()
        e += sub[sub["technology"].isin(ELECTRIC_TECHS)]["pkm_fleet"].sum()
    return e / t if t > 0 else 0.0
EV_SHARE_2025 = _current_ev_share()

print(f"Population ............. {POP:>12,} hab")
print(f"Total pkm 2025 ......... {total_pkm_2025/1e9:>8.2f} Gpkm")
print(f"Auto share 2025 ........ {pkm_by_mode_2025['Auto']/total_pkm_2025:>8.1%}")
print(f"EV share 2025 (motorised){100*EV_SHARE_2025:>8.2f} %")
if fallback_notes:
    print("Input fallbacks applied:")
    for note in fallback_notes:
        print("  -", note)
print("Baseline pkm by mode:")
for m in MODES:
    print(f"  {m:<8} {pkm_by_mode_2025[m]:>15,.0f}")


Population .............      735,332 hab
Total pkm 2025 .........     5.90 Gpkm
Auto share 2025 ........    70.6%
EV share 2025 (motorised)    0.91 %
Baseline pkm by mode:
  Walking      438,948,130
  Auto       4,166,221,037
  Bus          516,517,981
  Moto         189,199,982
  Bike          94,599,991
  Train        496,677,196


## 3 · Emission factors — operational vs embodied

For every vehicle technology (Car_ICE_Gasoline, Car_BEV, Bus_BEV, …) the
`vehicle_emissions_package` returns a gCO₂eq/pkm breakdown with these columns:

- **Operational** (respond to the electricity grid through WTT):
  `TTW_gCO2eq_per_pkm`, `WTT_gCO2eq_per_pkm`
- **Embodied** (respond to L_Embodied):
  `B1_flu_pkm`, `B1_mfg_pkm`, `B1_asm_pkm`, `B1_tra_pkm`,
  `B2_inf_pkm`, `B2_svc_pkm`, `B3_dis_pkm`

We aggregate each technology up to its parent modal class by weighting
`tech_share_pkm` within the mode. We also build the *BEV-variant* mapping
that says "what this technology becomes if it is electrified" — so we can
linearly ramp from today's fossil mix to a 100 %-electrified 2050 fleet.

In [39]:
OPERATIONAL_COLS = ["TTW_gCO2eq_per_pkm", "WTT_gCO2eq_per_pkm"]
EMBODIED_COLS    = ["B1_flu_pkm", "B1_mfg_pkm", "B1_asm_pkm", "B1_tra_pkm",
                    "B2_inf_pkm", "B2_svc_pkm", "B3_dis_pkm"]

tech_to_bev = {
    "Walk": "Walk", "Bike_Standalone": "Bike_Standalone", "eBike": "eBike",
    "BikeSharing_Standalone": "BikeSharing_Standalone", "eBikeSharing": "eBikeSharing",
    "Moped_ICE": "Moped_BEV", "Moped_BEV": "Moped_BEV",
    "Car_ICE_Gasoline": "Car_BEV", "Car_ICE_Diesel": "Car_BEV",
    "Car_PHEV": "Car_BEV", "Car_BEV": "Car_BEV",
    "Taxi_ICE": "Taxi_BEV", "Taxi_PHEV": "Taxi_BEV", "Taxi_BEV": "Taxi_BEV",
    "Bus_ICE": "Bus_BEV", "Bus_PHEV": "Bus_BEV", "Bus_BEV": "Bus_BEV",
    "Rail_EV": "Rail_EV",
}

def _ef_by_mode(electric_wtt):
    """Return {'base_op':dict, 'base_emb':dict, 'bev_op':dict, 'bev_emb':dict}.
    Each dict maps mode -> gCO2eq/pkm at the given grid WTT.
    """
    df = build_df_merged_final(electric_wtt=electric_wtt).copy()
    df["EF_op"]  = df[OPERATIONAL_COLS].sum(axis=1)
    df["EF_emb"] = df[EMBODIED_COLS].sum(axis=1)
    op_t  = dict(zip(df["tech"], df["EF_op"]))
    emb_t = dict(zip(df["tech"], df["EF_emb"]))

    def _agg(mapper):
        op_m, emb_m = {}, {}
        for mode in MODES:
            sub = fleet[fleet["modal_mode"] == mode]
            if sub.empty:
                op_m[mode] = emb_m[mode] = 0.0; continue
            tm = sub["technology"].map(mapper)
            op_m[mode]  = float((sub["tech_share_pkm"] * tm.map(op_t)).sum())
            emb_m[mode] = float((sub["tech_share_pkm"] * tm.map(emb_t)).sum())
        return op_m, emb_m
    bo, be = _agg(lambda t: t)                       # today's mix (base)
    eo, ee = _agg(lambda t: tech_to_bev.get(t, t))   # fully electrified
    return {'base_op': bo, 'base_emb': be, 'bev_op': eo, 'bev_emb': ee}
print("EF helper ready.")


EF helper ready.


## 4 · Grid scenarios (exogenous)

Three IEA-style electricity-grid trajectories are used as an exogenous
backdrop. They are piecewise-linear interpolations through a handful of
checkpoints (gCO₂eq/kWh at key years).

| Scenario | 2025 | 2030 | 2040 | 2050 | Narrative |
|---|---|---|---|---|---|
| STEPS | 115 | 75 | 45 | 25 | Current national policies |
| APS   | 115 | 40 | 10 |  5 | Announced pledges honoured |
| NZE   | 115 |  0 |  0 |  0 | Net-Zero Emissions by 2030 |

For each distinct grid carbon-intensity value we pre-compute the full
`{base_op, base_emb, bev_op, bev_emb}` dictionary once. This is the most
expensive operation in the whole notebook, so doing it up-front saves
massive time during the sweep.

In [40]:
GRID_SCENARIOS = {
    "STEPS": {2025: 115, 2030: 75, 2040: 45, 2050: 25},
    "APS":   {2025: 115, 2030: 40, 2040: 10, 2050: 5},
    "NZE":   {2025: 115, 2030: 0,  2035: 0,  2040: 0,  2050: 0},
}

def _grid_trajectory(points):
    ys = sorted(points); out = {}
    for y in YEARS:
        if y <= ys[0]:   out[y] = float(points[ys[0]])
        elif y >= ys[-1]: out[y] = float(points[ys[-1]])
        else:
            for i in range(len(ys)-1):
                if ys[i] <= y <= ys[i+1]:
                    t = (y - ys[i]) / (ys[i+1] - ys[i])
                    out[y] = float(points[ys[i]] + t*(points[ys[i+1]] - points[ys[i]]))
                    break
    return out

grid_trajectories = {name: {y: round(v,1) for y,v in _grid_trajectory(p).items()}
                     for name,p in GRID_SCENARIOS.items()}

_all_grid_values = sorted({v for t in grid_trajectories.values() for v in t.values()})
print(f"Distinct grid carbon-intensity values to evaluate: {len(_all_grid_values)}")

t0 = time.time()
ef_by_grid_value = {g: _ef_by_mode(electric_wtt=float(g)) for g in _all_grid_values}
print(f"EF cache built in {time.time()-t0:.1f} s  ({len(ef_by_grid_value)} grid values)")


Distinct grid carbon-intensity values to evaluate: 52
EF cache built in 0.5 s  (52 grid values)


## 5 · Lever definitions

The decarbonisation space is parameterised by **five levers**. Four of them
are policy-actionable within the region (L₁…L_Embodied); the fifth (Grid)
is exogenous — set by supra-national electricity-sector policy.

The table below is the authoritative definition; the next subsection walks
through each lever in depth.

| Lever | Short name | What it controls | Domain | Step |
|---|---|---|---|---|
| **L₁** | EV share | Share of *motorised* pkm delivered by BEVs in 2050 | [0 %, 100 %] | 10 pp (11 values) |
| **L₂** | Modal shift | Annual compound rate at which Auto pkm migrate to non-car modes | [0 %/yr, +6 %/yr] | 0.1 pp (61 values) |
| **L₃** | Demand reduction | Annual compound rate at which Auto & Moto pkm shrink (telework, 15-min city, avoided trips) | [0 %/yr, −6 %/yr] | 0.1 pp (61 values) |
| **L_Embodied** | Embodied-LCA cut | Total reduction of embodied EF (B1+B2+B3) achieved by 2050, linearly interpolated | [0 %, 100 %] | 10 pp (11 values) |
| **Grid** | Grid scenario | Electricity WTT trajectory | {STEPS, APS, NZE} | 3 |

**Total combinatorial space: 11 × 61 × 61 × 11 × 3 = 1 350 723 scenarios.**

### Why this step-size split?

The user-requested step was 0.1 pp uniformly. That is *scientifically*
meaningful for the continuous per-annum rates L₂ and L₃ — the difference
between 2.0 %/yr and 2.1 %/yr of modal shift is a real, measurable policy
difference. For L₁ and L_Embodied, both of which represent a *cumulative
share* with a 100 pp domain, a 0.1 pp step would inflate the hypercube by
$10\,000\times$ (→ 13.5 × 10⁹ rows) with negligible additional resolution.
We therefore adopt a 10 pp step for these two, which still matches the
granularity every Gipuzkoa policy roadmap uses in practice (decadal EV
targets are quoted in 10 pp increments). If finer resolution is ever
needed, the vectorised pipeline below re-runs in <30 s with step 1 pp
(→ 101 × 61 × 61 × 101 × 3 ≈ 113 M rows) — just edit the `L1_VALUES` and
`L_EMB_VALUES` arrays.

### 5.1 — **L₁ · EV share of motorised pkm (by 2050)**

**Physical meaning.** Out of every passenger-kilometre travelled by a
*motorised* mode (Auto + Moto + Bus) in 2050, what fraction is supplied by
battery-electric powertrains? `L₁ = 0.4` means 40 % of motorised pkm in
2050 come from BEVs; the remaining 60 % still use the 2025 fossil-heavy
technology mix.

**Time profile.** We assume a **linear ramp** between today's actual EV
share (≈ 3 %) and the 2050 target value:

$$\alpha(L_1, y) \;=\; \text{EV}_{2025} + (L_1 - \text{EV}_{2025}) \cdot \frac{y - 2025}{25}$$

For every mode $m$ and every year $y$ the blended operational EF is

$$\text{EF}^{\text{op}}_{m,y} \;=\; (1-\alpha)\,\text{EF}^{\text{op,base}}_{m} + \alpha\,\text{EF}^{\text{op,BEV}}_{m,y}$$

Note that $\text{EF}^{\text{op,BEV}}_{m,y}$ itself depends on the grid
carbon-intensity at year $y$, so L₁ interacts with Grid (see §6).

**What L₁ does NOT do.** It does **not** change the *amount* of pkm, nor
how they are distributed among modes. It only re-assigns the propulsion
technology of each motorised pkm. A high L₁ can still yield high
absolute emissions if demand (L₃) stays high and the grid (Grid) stays dirty.


### 5.2 — **L₂ · Modal shift (annual compound rate)**

**Physical meaning.** Each year, a fraction **L₂** of the *non-captive*
Auto pkm is reallocated to non-car modes (Walking, Bike, Bus, Train). A
captive-Auto fraction (18 %) is carved out up-front — rural users, PRM, and
freight-like private trips that cannot realistically shift — and only the
remainder is eligible for reassignment.

**How the shifted pkm are redistributed.** We use a distance-band
substitution matrix derived from regional OD-survey micro-data:

|              | Walking | Bike | Bus | Train |
|--------------|---------|------|-----|-------|
| Short (22 %) | 0.45    | 0.40 | 0.15| 0.00  |
| Medium (48%) | 0.05    | 0.35 | 0.45| 0.15  |
| Long  (30 %) | 0.00    | 0.05 | 0.25| 0.70  |

So if L₂ = 0.03 (3 %/yr) and 10 000 Auto pkm are shifted in a given year,
2 200 are short, 4 800 medium, 3 000 long; these are then split row-wise.

**Compounding.** L₂ acts **every year**: the pool of Auto pkm shrinks, so
in absolute pkm the yearly shift decays exponentially. By 2050, with
L₂ = 3 %/yr applied over 25 compounding years, the cumulative Auto-to-shift
pkm equals $1 - (1-0.82 \cdot 0.03)^{25} \approx 46$ % of the original
Auto pkm (the 0.82 factor is the non-captive fraction).

**Interaction with L₃.** Both L₂ and L₃ modify the pkm state. They commute
almost perfectly *in expectation*, but the simulator applies them in a
fixed order (L₃ first → L₂ second, §8) so the output is deterministic.
Because L₃ shrinks the Auto pkm pool first, a given L₂ moves fewer pkm
when L₃ is strongly negative — this is a mild but real anti-synergy.


### 5.3 — **L₃ · Demand reduction (annual compound rate)**

**Physical meaning.** Each year, the total pkm delivered by *motorised
private* modes (Auto + Moto) shrinks by L₃:

$$\text{pkm}^{\text{Auto}}_{y+1} = \text{pkm}^{\text{Auto}}_{y} \cdot (1 + L_3),\quad L_3 \le 0$$

Concretely this captures the combined effect of avoided trips due to
remote work, the 15-minute city, consolidated shopping, e-commerce, trip
chaining, and rebound-free adoption of active modes for short trips.

**What is NOT demand reduction.** Walking, biking, bus and train pkm are
NOT shrunk by L₃ — they are treated as "genuine mobility" and often grow
via modal shift (L₂). L₃ is specifically about **avoided car pkm**, not
avoided journeys writ large.

**Compounding.** L₃ is a geometric rate. L₃ = −3 %/yr applied for 25 years
yields a factor of $(1-0.03)^{25} \approx 0.467$ — the Auto pkm pool in
2050 is 47 % of its 2025 value.

**Interaction with L₂.** After L₃ shrinks the pool, L₂ shifts *what
remains*. So the amount of pkm that reach non-car modes under L₂ is
attenuated by L₃. From the atmosphere's perspective this is fine — fewer
Auto pkm to begin with means fewer emissions regardless — but it means the
two levers are **partial substitutes**, not pure additives (see §6).

**Interaction with L₁ and L_Embodied.** L₃ directly shrinks the base over
which L₁ and L_Embodied operate: if there are no cars to be driven, the
fact that they would have been electric matters less.


### 5.4 — **L_Embodied · Embodied-LCA cut (by 2050)**

**Physical meaning.** A single number — the fraction by which the embodied
emission factor (B1 manufacturing + B2 infrastructure + B3 disposal) is
lowered by 2050, relative to the 2025 value. Mechanisms lumped under this
lever include:

- Recycled-content steel, aluminium and battery materials (B1).
- Green-cement roads, reduced road build-out (B2).
- Circular-economy disposal / second-life batteries (B3).

**Time profile.** Like L₁, we apply a **linear ramp**:

$$\text{emb\_scale}(L_{\\text{Emb}}, y) \;=\; 1 - L_{\\text{Emb}} \cdot \frac{y - 2025}{25}$$

The embodied EF at year $y$ is

$$\text{EF}^{\text{emb}}_{m,y} \;=\; \big[(1-\alpha)\,\text{EF}^{\text{emb,base}}_{m} + \alpha\,\text{EF}^{\text{emb,BEV}}_{m}\big]\;\cdot\;\text{emb\_scale}(L_{\\text{Emb}}, y)$$

so L_Embodied *multiplicatively* deflates whatever embodied baseline the
mode–EV mix delivers.

**Why 0 → 100 pp range.** L_Embodied = 100 % is the theoretical asymptote
where by 2050 every new vehicle, road metre, and battery cell is net-zero
embodied. This is aspirational and probably physically unattainable in full,
but we sweep the full range to expose the shape of the feasibility frontier.

**Interaction with L₁.** L_Embodied acts on the *blended* embodied EF, so
a high L₁ (more BEVs) makes L_Embodied act on a larger, battery-dominated
pool. Conversely a low L₁ means L_Embodied is cutting mostly internal-
combustion-vehicle manufacturing emissions.

**Interaction with L₂ and L₃.** Because embodied EF is *per pkm*, it scales
linearly with the pkm counted in the Kaya identity. L₂ re-routes pkm to
modes with *different* embodied EFs (rail and buses have much lower
B₁+B₂+B₃ per pkm than private cars), so L_Embodied always stacks
constructively with L₂. L₃ simply reduces the pkm the embodied EF is
applied to, so the two multiply.


### 5.5 — **Grid · Electricity-grid decarbonisation (exogenous)**

**Physical meaning.** The carbon-intensity of the Spanish/Basque
electricity grid, in gCO₂eq/kWh, at each year between 2025 and 2050. This
lever is exogenous — decided by EU, national and REE/Iberdrola policy —
so we sweep three representative IEA pathways (§4) rather than a
continuous range.

**What Grid touches.** Only the *WTT* component of the operational EF for
electrified technologies. ICE-gasoline and ICE-diesel EFs are invariant
under Grid.

**Interaction with L₁.** This is the headline interaction of the whole
study. The benefit of switching a passenger-kilometre from ICE to BEV
is $(\text{EF}^{\text{op,ICE}} - \text{EF}^{\text{op,BEV}}(\text{Grid}))$.
If the grid is dirty (STEPS: 115 gCO₂/kWh today → 25 g in 2050), that
difference is smaller. If the grid is clean (NZE: 0 from 2030), it
maximises. Concretely:

- L₁ × STEPS → modest benefit; EVs are still cleaner than ICE, but not by
  the factor-of-10 margin that is often quoted.
- L₁ × NZE → full benefit; every BEV pkm is ≈ zero operational emissions
  from 2030 onwards.

**No interaction with L₂, L₃, L_Embodied.** Grid does not directly change
pkm or modal shares, and acts only on operational EF (not embodied).


## 6 · Lever interactions — when each acts on the other

The matrix below summarises *who depends on whom*. A `↘` means the row
lever **changes the base** on which the column lever operates; a `↗` means
the row lever amplifies the effect of the column lever.

|                 | **L₁**     | **L₂**       | **L₃**       | **L_Emb**    | **Grid**     |
|-----------------|------------|--------------|--------------|--------------|--------------|
| **L₁**          | —          | —            | —            | ↗ (shifts embodied baseline) | ↗ (larger benefit on clean grid) |
| **L₂**          | ↘ (less Auto → less EV-potential) | — | anti-synergy (acts on residual Auto after L₃) | ↗ (rail/bus have lower embodied EF)| — |
| **L₃**          | ↘ (shrinks the pool EVs run on) | ↘ (shrinks the pool to shift) | — | ↘ (less pkm → less embodied) | — |
| **L_Embodied**  | —          | —            | —            | —            | —            |
| **Grid**        | ↗ (see above) | — | — | — | —            |

### Order of operations (per year, deterministic)

1. **Apply L₃** — shrink Auto & Moto pkm by $(1 + L_3)$.
2. **Apply L₂** — shift an L₂ fraction of non-captive Auto pkm to non-car
   modes using the distance-band substitution matrix.
3. **Compute $\alpha(L_1, y)$** — EV blend for the year.
4. **Compute $\text{emb\_scale}(L_\text{Emb}, y)$** — embodied ramp.
5. **Pick grid EF dictionary** for the year's grid intensity.
6. **Emissions = Σ pkm × [EF_op + EF_emb × emb_scale] × 1e-6 / POP.**

This ordering is important for the anti-synergy between L₂ and L₃: because
L₃ runs first, L₂ acts on the already-shrunken Auto pool.

### Scalar decomposition that enables fast vectorisation

Once we fix (L₂, L₃, Grid), the per-year emission can be written

$$E_y \;=\; K_0(y) + \alpha(L_1,y)\,K_1(y) + \text{emb\_scale}(L_\text{Emb},y)\,K_2(y) + \alpha(L_1,y)\,\text{emb\_scale}(L_\text{Emb},y)\,K_3(y)$$

where $K_0..K_3$ are four pre-computable scalars per (L₂, L₃, Grid, year).
This **linear-in-L₁** and **linear-in-L_Embodied** property is what lets
us skip building the full 5-D hypercube. The derivation is straightforward
from the Kaya identity and is implemented in §9.

## 7 · Sweep definition

In [41]:
# ─── Sweep values ──────────────────────────────────────────────────────────
L1_VALUES     = np.round(np.arange(0.0, 1.0 + 1e-9, 0.10), 2)   # 0..1 step 0.10  → 11
L2_VALUES     = np.round(np.arange(0.0, 0.06 + 1e-9, 0.001), 4) # 0..0.06 step 0.001 → 61
L3_VALUES     = np.round(np.arange(0.0, -0.06 - 1e-9, -0.001),4)# 0..-0.06 step -0.001 → 61
L_EMB_VALUES  = np.round(np.arange(0.0, 1.0 + 1e-9, 0.10), 2)   # 0..1 step 0.10  → 11
GRID_VALUES   = ["STEPS", "APS", "NZE"]

n1, n2, n3, ne, ng = len(L1_VALUES), len(L2_VALUES), len(L3_VALUES), len(L_EMB_VALUES), len(GRID_VALUES)
N_SCEN = n1 * n2 * n3 * ne * ng

print(f"L1           : {n1:>4d} values  [{L1_VALUES[0]:.2f} .. {L1_VALUES[-1]:.2f}]")
print(f"L2           : {n2:>4d} values  [{L2_VALUES[0]*100:.2f}% .. {L2_VALUES[-1]*100:.2f}%] /yr")
print(f"L3           : {n3:>4d} values  [{L3_VALUES[0]*100:.2f}% .. {L3_VALUES[-1]*100:.2f}%] /yr")
print(f"L_Embodied   : {ne:>4d} values  [{L_EMB_VALUES[0]:.2f} .. {L_EMB_VALUES[-1]:.2f}]")
print(f"Grid         : {ng:>4d} values  {GRID_VALUES}")
print(f"────────────────")
print(f"Total rows   : {N_SCEN:>9,}")


L1           :   11 values  [0.00 .. 1.00]
L2           :   61 values  [0.00% .. 6.00%] /yr
L3           :   61 values  [0.00% .. -6.00%] /yr
L_Embodied   :   11 values  [0.00 .. 1.00]
Grid         :    3 values  ['STEPS', 'APS', 'NZE']
────────────────
Total rows   : 1,350,723


## 8 · Step 1 — Vectorised pkm evolution

For every (L₂, L₃) pair we simulate the pkm state year by year. The state
is a 4-D array `pkm_state[mode, i_L2, i_L3]` (shape 6 × 61 × 61). At each
year we

1. grow/shrink Auto & Moto via L₃,
2. apply modal shift via L₂ (vectorised over all (L₂, L₃) combos).

We store the full trajectory `pkm_trajectory[year, mode, i_L2, i_L3]`.
Shape: 26 × 6 × 61 × 61 ≈ 580 k floats ≈ 4.6 MB. Negligible.

In [42]:
# ─── Constants from the behavioural modal-shift engine ─────────────────────
CAPTIVE_AUTO    = 0.18
AUTO_DIST_BANDS = {'short': 0.22, 'medium': 0.48, 'long': 0.30}
SUBST_MATRIX = {
    'short':  {'Walking': 0.45, 'Bike': 0.40, 'Bus': 0.15, 'Train': 0.00},
    'medium': {'Walking': 0.05, 'Bike': 0.35, 'Bus': 0.45, 'Train': 0.15},
    'long':   {'Walking': 0.00, 'Bike': 0.05, 'Bus': 0.25, 'Train': 0.70},
}
DEST_MODES = ['Walking', 'Bike', 'Bus', 'Train']
MODE_IDX   = {m: i for i, m in enumerate(MODES)}

# Pre-compute the aggregate share received by each destination mode per unit
# of Auto pkm shifted. This collapses the 3-band × 4-dest matrix into a
# single {mode -> weight} dict.
_SHIFT_WEIGHT = {m: 0.0 for m in DEST_MODES}
for band, bshare in AUTO_DIST_BANDS.items():
    for dest, p in SUBST_MATRIX[band].items():
        _SHIFT_WEIGHT[dest] += bshare * p
SHIFT_WEIGHT_VEC = np.array([_SHIFT_WEIGHT.get(m, 0.0) for m in MODES])   # (6,)

def compute_pkm_trajectory(L2_arr, L3_arr):
    """Return pkm_trajectory of shape (N_YEARS, N_MODES, |L2|, |L3|)."""
    n2, n3 = len(L2_arr), len(L3_arr)
    pkm = np.zeros((N_MODES, n2, n3), dtype=np.float64)
    for i, m in enumerate(MODES):
        pkm[i] = pkm_by_mode_2025[m]                                     # broadcast

    traj = np.zeros((N_YEARS, N_MODES, n2, n3), dtype=np.float64)
    traj[0] = pkm

    auto_i, moto_i = MODE_IDX['Auto'], MODE_IDX['Moto']
    L2_grid = L2_arr[:, None]        # (n2, 1)
    L3_grid = L3_arr[None, :]        # (1, n3)
    growth  = 1.0 + L3_grid          # (1, n3) — shared by Auto & Moto

    for iy in range(1, N_YEARS):
        # Step 1 — L3: compound demand reduction
        pkm[auto_i] *= growth
        pkm[moto_i] *= growth

        # Step 2 — L2: modal shift (vectorised over n2, n3)
        shiftable   = pkm[auto_i] * (1.0 - CAPTIVE_AUTO)
        total_shift = shiftable * L2_grid                                # (n2, n3)
        pkm[auto_i] -= total_shift
        for i_mode, w in enumerate(SHIFT_WEIGHT_VEC):
            if w > 0.0:
                pkm[i_mode] += total_shift * w

        traj[iy] = pkm
    return traj

t0 = time.time()
pkm_traj = compute_pkm_trajectory(L2_VALUES, L3_VALUES)
print(f"pkm_trajectory shape {pkm_traj.shape}   built in {time.time()-t0:.2f} s")
print(f"Memory: {pkm_traj.nbytes/1e6:.1f} MB")


pkm_trajectory shape (26, 6, 61, 61)   built in 0.00 s
Memory: 4.6 MB


## 9 · Step 2 — Build EF arrays per (L₁, Grid, year, mode)

The alpha-blend for operational and embodied EFs depends on:
- $\alpha(L_1, y)$: linear ramp from `EV_SHARE_2025` to L₁ across 25 yrs.
- Grid scenario: picks which element of the pre-cached EF dictionary applies.

We build two 4-D arrays:
- `ef_op_arr[iL1, iG, iy, im]` — gCO₂eq/pkm operational,
- `ef_emb_arr[iL1, iG, iy, im]` — gCO₂eq/pkm embodied (before L_Embodied).

Shape: 11 × 3 × 26 × 6 ≈ 5 k floats. Tiny.

In [43]:
FRAC = (YEARS - 2025) / 25.0                                            # (26,)

def build_ef_arrays():
    ef_op  = np.zeros((n1, ng, N_YEARS, N_MODES), dtype=np.float64)
    ef_emb = np.zeros_like(ef_op)
    for iL1, L1 in enumerate(L1_VALUES):
        alpha_yr = EV_SHARE_2025 + (L1 - EV_SHARE_2025) * FRAC              # (26,)
        alpha_yr = np.clip(alpha_yr, 0.0, 1.0)
        for iG, gname in enumerate(GRID_VALUES):
            traj = grid_trajectories[gname]
            for iy, y in enumerate(YEARS):
                ef = ef_by_grid_value[round(traj[y], 1)]
                a = alpha_yr[iy]
                for im, m in enumerate(MODES):
                    ef_op [iL1, iG, iy, im] = (1.0 - a) * ef['base_op'][m]  + a * ef['bev_op'][m]
                    ef_emb[iL1, iG, iy, im] = (1.0 - a) * ef['base_emb'][m] + a * ef['bev_emb'][m]
    return ef_op, ef_emb

t0 = time.time()
ef_op_arr, ef_emb_arr = build_ef_arrays()
print(f"ef_op_arr  {ef_op_arr.shape}   ef_emb_arr {ef_emb_arr.shape}")
print(f"Built in {time.time()-t0:.2f} s")


ef_op_arr  (11, 3, 26, 6)   ef_emb_arr (11, 3, 26, 6)
Built in 0.00 s


## 10 · Step 3 — Contract into the 4-D part arrays

For every (L₁, L₂, L₃, Grid, year) we compute

$$P_\text{op}(\cdot) = \sum_m \text{pkm}(m, L_2, L_3, y) \cdot \text{EF}^\text{op}_m(L_1, G, y)$$

and the analogous $P_\text{emb}$. L_Embodied does not appear yet — it
enters as a per-year linear factor later.

Shape of `P_op` and `P_emb`: 11 × 61 × 61 × 3 × 26 ≈ 6.5 M floats (52 MB
as float64, 26 MB as float32). We stay in float64 for the contraction and
down-cast at the end.

In [44]:
# pkm_traj : (N_YEARS, N_MODES, n2, n3)
# ef_op_arr: (n1, ng, N_YEARS, N_MODES)
# We want  P_op[iL1, i2, i3, iG, iy] = Σ_m pkm_traj[iy, m, i2, i3] * ef_op_arr[iL1, iG, iy, m]

t0 = time.time()
# einsum: y m 2 3 , 1 g y m -> 1 2 3 g y   (axis letters = {L1, L2, L3, Grid, year, mode})
P_op  = np.einsum('ymab,lgym->labgy', pkm_traj, ef_op_arr,  optimize=True)
P_emb = np.einsum('ymab,lgym->labgy', pkm_traj, ef_emb_arr, optimize=True)
print(f"P_op  shape {P_op.shape}")
print(f"P_emb shape {P_emb.shape}")
print(f"Contraction: {time.time()-t0:.2f} s   memory={2*P_op.nbytes/1e6:.1f} MB")


P_op  shape (11, 61, 61, 3, 26)
P_emb shape (11, 61, 61, 3, 26)
Contraction: 0.01 s   memory=51.1 MB


## 11 · Step 4 — Apply L_Embodied and assemble per-year emissions

$$E_y(L_1, L_2, L_3, L_\text{Emb}, G) \;=\; \frac{1}{\text{POP} \cdot 10^6}\left[P_\text{op} + \text{emb\_scale}(L_\text{Emb}, y) \cdot P_\text{emb}\right]$$

We loop over the 11 L_Embodied values (cheap) and stack the results into a
5-D tensor. Shape: 11 × 61 × 61 × 3 × 11 × 26 ≈ 72 M floats ≈ 290 MB in
float64. We down-cast to float32 as we go (→ 145 MB) and the row-wise
cumulative sum reduces the parquet to ~30 MB.

In [45]:
t0 = time.time()

# emb_scale(L_Emb, y) = 1 - L_Emb * frac(y)     shape (ne, N_YEARS)
emb_scale = 1.0 - L_EMB_VALUES[:, None] * FRAC[None, :]                 # (ne, 26)

# Allocate per-capita per-year array: (n1, n2, n3, ng, ne, N_YEARS)  float32
per_cap = np.empty((n1, n2, n3, ng, ne, N_YEARS), dtype=np.float32)

INV_NORM = 1.0 / (POP * 1e6)  # gCO2 → tCO2 / hab (pkm * g/pkm = g; /1e6 = t; /POP = per cap)

for ie in range(ne):
    # per_year_5D[iL1, i2, i3, iG, iy] in tCO2eq/cap
    em_year = (P_op + emb_scale[ie][None, None, None, None, :] * P_emb) * INV_NORM
    per_cap[:, :, :, :, ie, :] = em_year.astype(np.float32)

print(f"per_cap shape {per_cap.shape}   memory {per_cap.nbytes/1e6:.1f} MB")
print(f"Done in {time.time()-t0:.2f} s")


per_cap shape (11, 61, 61, 3, 11, 26)   memory 140.5 MB
Done in 0.14 s


## 12 · Step 5 — Cumulative absolute emissions

$$\text{cum\_tCO2eq\_total}(\cdot) = \text{POP} \cdot \sum_{y=2025}^{2050} \text{per\_cap}_y(\cdot)$$

In [46]:
cum_total = (per_cap.sum(axis=-1, dtype=np.float64) * POP).astype(np.float32)
print(f"cum_total shape {cum_total.shape}   memory {cum_total.nbytes/1e6:.1f} MB")
print(f"Range: {cum_total.min():,.0f} .. {cum_total.max():,.0f}  tCO2eq")


cum_total shape (11, 61, 61, 3, 11)   memory 5.4 MB
Range: 5,486,269 .. 18,299,340  tCO2eq


## 13 · Step 6 — Assemble the tidy DataFrame and persist

Columns:
- `L1_EV`, `L2_modal`, `L3_demand`, `L_Embodied` — lever coordinates
- `Grid` — categorical string
- `cum_tCO2eq_total` — absolute cumulative emissions 2025→2050
- `per_cap_2025` … `per_cap_2050` — 26 columns, tCO₂eq/hab/yr

Row order: the natural nesting `(L1, L2, L3, Grid, L_Embodied)`, which
matches the memory layout of `per_cap` and `cum_total`. We use Parquet
(with zstd compression) for compact storage; a 1 000-row CSV preview is
also written for quick inspection.

In [47]:
t0 = time.time()

# Flatten axis order: (L1, L2, L3, Grid, L_Embodied)  → same as per_cap
# per_cap dims already are (n1, n2, n3, ng, ne, N_YEARS) → perfect.
N = n1 * n2 * n3 * ng * ne
assert N == N_SCEN, "sanity: N should equal N_SCEN"

# Build the five lever index arrays via broadcasting once.
i1 = np.repeat(np.arange(n1), n2*n3*ng*ne)
i2 = np.tile  (np.repeat(np.arange(n2), n3*ng*ne), n1)
i3 = np.tile  (np.repeat(np.arange(n3), ng*ne),    n1*n2)
ig = np.tile  (np.repeat(np.arange(ng), ne),       n1*n2*n3)
ie = np.tile  (np.arange(ne),                      n1*n2*n3*ng)

df = pd.DataFrame({
    "L1_EV":      L1_VALUES[i1].astype(np.float32),
    "L2_modal":   L2_VALUES[i2].astype(np.float32),
    "L3_demand":  L3_VALUES[i3].astype(np.float32),
    "Grid":       pd.Categorical([GRID_VALUES[k] for k in ig], categories=GRID_VALUES),
    "L_Embodied": L_EMB_VALUES[ie].astype(np.float32),
    "cum_tCO2eq_total": cum_total.reshape(-1),
})

# Attach the 26 per-year per-capita columns.
per_cap_flat = per_cap.reshape(N, N_YEARS)
for iy, y in enumerate(YEARS):
    df[f"per_cap_{y}"] = per_cap_flat[:, iy]

print(f"DataFrame assembled in {time.time()-t0:.1f} s")
print(f"Rows: {len(df):,}   Columns: {len(df.columns)}")
print(f"In-memory size: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")


DataFrame assembled in 0.3 s
Rows: 1,350,723   Columns: 32
In-memory size: 168.8 MB


In [48]:
t0 = time.time()
df.to_parquet(PARQUET_PATH, engine="pyarrow", compression="zstd", index=False)
print(f"Parquet written: {PARQUET_PATH}  ({PARQUET_PATH.stat().st_size/1e6:.1f} MB)   in {time.time()-t0:.1f} s")

# A human-readable 1 000-row sample for immediate inspection
df.head(1000).to_csv(CSV_PREVIEW, index=False)
print(f"CSV preview:    {CSV_PREVIEW}  ({CSV_PREVIEW.stat().st_size/1e3:.0f} KB, first 1 000 rows)")


Parquet written: scenarios_master_toolkit.parquet  (148.4 MB)   in 1.0 s
CSV preview:    scenarios_master_toolkit_preview.csv  (305 KB, first 1 000 rows)


## 14 · Summary statistics & sanity checks

Three quick checks to confirm the dataset is sensible:

1. **Baseline** (all levers off, STEPS grid): cumulative should land near the
   historical 2025 per-capita figure × 26.
2. **Extreme decarbonisation** (L₁=1, L₂=6 %/yr, L₃=−6 %/yr, L_Emb=1, NZE):
   should be dramatically lower.
3. **Budget compliance**: how many scenarios fall inside the 1.5 °C
   (8.82 tCO₂eq/cap cumulative) and 2 °C (42.82 tCO₂eq/cap cumulative) budgets.

In [49]:
# ─── (1) Baseline row ──────────────────────────────────────────────────────
baseline = df[(df.L1_EV == 0.0) & (df.L2_modal == 0.0) & (df.L3_demand == 0.0)
              & (df.L_Embodied == 0.0) & (df.Grid == "STEPS")].iloc[0]
print(f"BASELINE (all off, STEPS)")
print(f"  cum per-cap 2025-2050: {baseline.filter(like='per_cap_').sum():.2f} tCO2eq/cap")
print(f"  cum absolute         : {baseline['cum_tCO2eq_total']/1e6:.2f} MtCO2eq")

# ─── (2) Extreme decarb row ────────────────────────────────────────────────
extreme = df[(df.L1_EV == 1.0) & (df.L2_modal == 0.06) & (df.L3_demand == -0.06)
             & (df.L_Embodied == 1.0) & (df.Grid == "NZE")].iloc[0]
print(f"\nEXTREME (L1=1, L2=+6%/yr, L3=-6%/yr, L_Emb=1, NZE)")
print(f"  cum per-cap 2025-2050: {extreme.filter(like='per_cap_').sum():.2f} tCO2eq/cap")
print(f"  cum absolute         : {extreme['cum_tCO2eq_total']/1e6:.2f} MtCO2eq")

# ─── (3) Budget compliance ─────────────────────────────────────────────────
PC_COLS = [f"per_cap_{y}" for y in YEARS]
df["cum_per_cap"] = df[PC_COLS].sum(axis=1)
BUDGET_15C = 8.82
BUDGET_2C  = 42.82
n_15 = (df["cum_per_cap"] <= BUDGET_15C).sum()
n_2  = (df["cum_per_cap"] <= BUDGET_2C ).sum()
print(f"\nBUDGET COMPLIANCE")
print(f"  ≤ 1.5 °C budget  ({BUDGET_15C:5.2f} tCO2eq/cap): {n_15:>9,} / {len(df):,}  ({100*n_15/len(df):5.2f} %)")
print(f"  ≤ 2.0 °C budget  ({BUDGET_2C :5.2f} tCO2eq/cap): {n_2 :>9,} / {len(df):,}  ({100*n_2/len(df):5.2f} %)")


BASELINE (all off, STEPS)
  cum per-cap 2025-2050: 24.89 tCO2eq/cap
  cum absolute         : 18.30 MtCO2eq

EXTREME (L1=1, L2=+6%/yr, L3=-6%/yr, L_Emb=1, NZE)
  cum per-cap 2025-2050: 7.46 tCO2eq/cap
  cum absolute         : 5.49 MtCO2eq

BUDGET COMPLIANCE
  ≤ 1.5 °C budget  ( 8.82 tCO2eq/cap):    18,783 / 1,350,723  ( 1.39 %)
  ≤ 2.0 °C budget  (42.82 tCO2eq/cap): 1,350,723 / 1,350,723  (100.00 %)


## 15. Provenance

Every row in `scenarios_master_toolkit.parquet` is reproducible from:

- `Toolkit_Mobility_Pathways_Inputs.xlsx`
- `vehicle_emissions_package`
- `GRID_SCENARIOS` dictionary
- `L*_VALUES` sweep arrays
- this notebook's simulation order

For Gipuzkoa, `APPLY_GIP_COMPATIBILITY_PATCH=True` reproduces the original `Scenarios_Definition_Levers_ASI.ipynb` baseline despite known workbook differences. For a new lab, set that flag to `False` and provide complete VKT/occupancy and fleet data in the workbook.
